# 1. Conda 环境设置（Windows 版）

语言：[English](../en/01_environment_setup.ipynb) | **中文**

本 Notebook 创建或更新仓库定义的 `cofkit` Conda 环境，并把它注册为 Jupyter 内核。本原生 Windows 版使用普通 Python 单元格，不需要 PowerShell 或 WSL。请在本仓库的克隆目录中运行。

在适用处，注释掉的单元格展示对应的直接 Python 检查。

## 教程系列

1. **环境设置**（本 Notebook）
2. [首次构建 COF](02_first_cof_build.ipynb)
3. [拓扑、连接数组合与连接键示例](03_topologies_connectivities_and_linkages.ipynb)
4. [使用 Zeo++ 进行孔隙分析](04_zeopp_pore_analysis.ipynb)

请从仓库根目录启动 Jupyter，确保所有相对路径都能一致解析。普通 Python 单元格通过 `subprocess` 调用 Conda，因此无需在单元格之间保持任何 Shell 状态。

## 查看仓库中的环境定义

`environment.yml` 的第一行将环境名称固定为 `cofkit`。整个教程系列都使用这个名称。

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve()
environment_file = repo_root / "environment.yml"
if not environment_file.is_file():
    raise FileNotFoundError(
        "请从 cofkit 仓库根目录启动 Jupyter，然后重新运行此单元格。"
    )

print(f"仓库根目录：{repo_root}\n")
print("\n".join(environment_file.read_text(encoding="utf-8").splitlines()[:24]))

In [ ]:
# Python 等价实现（已注释）：查看环境名称。
# from pathlib import Path
# environment_name = next(
#     line.split(":", 1)[1].strip()
#     for line in Path("environment.yml").read_text().splitlines()
#     if line.startswith("name:")
# )
# print(environment_name)

## 创建或更新环境

这个单元格会先检查 `cofkit` CLI 是否已在 `PATH` 中可用。如果可用，单元格将成功退出且不修改环境，这在选择 **Run All**（全部运行）时尤其有用。否则，若 `cofkit` 环境不存在则创建它，存在则根据 `environment.yml` 更新；随后安装本地软件包而不替换 Conda 已解析的依赖，并为 Notebooks 2–4 注册 `Python (cofkit)` 内核。

> 首次运行时，环境依赖解析和 Open Babel 安装可能需要几分钟。

In [ ]:
import json
import os
from pathlib import Path
import shutil
import subprocess

cofkit_command = shutil.which("cofkit")
if cofkit_command:
    print(
        f"cofkit CLI 已可用，位置：{cofkit_command}；"
        "跳过环境和软件包设置。"
    )
else:
    repo_root = Path.cwd().resolve()
    environment_file = repo_root / "environment.yml"
    if not environment_file.is_file():
        raise FileNotFoundError(
            "请从 cofkit 仓库根目录启动 Jupyter，然后重新运行此单元格。"
        )

    conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
    if not conda:
        raise RuntimeError(
            "未找到 Conda。请从 Miniforge、Miniconda 或 Anaconda "
            "启动 Jupyter，然后重新运行此单元格。"
        )

    environment_info = json.loads(
        subprocess.run(
            [conda, "env", "list", "--json"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout
    )
    environment_exists = any(
        Path(path).name.casefold() == "cofkit"
        for path in environment_info["envs"]
    )
    if environment_exists:
        environment_command = [
            conda, "env", "update", "--name", "cofkit",
            "--file", str(environment_file), "--prune",
        ]
    else:
        environment_command = [
            conda, "env", "create", "--file", str(environment_file),
        ]
    subprocess.run(environment_command, cwd=repo_root, check=True)
    subprocess.run(
        [conda, "run", "-n", "cofkit", "python", "-m", "pip",
         "install", "--no-deps", "."],
        cwd=repo_root,
        check=True,
    )
    subprocess.run(
        [conda, "install", "--name", "cofkit", "--channel", "conda-forge",
         "--yes", "ipykernel"],
        cwd=repo_root,
        check=True,
    )
    subprocess.run(
        [conda, "run", "-n", "cofkit", "python", "-m", "ipykernel",
         "install", "--user", "--name", "cofkit",
         "--display-name", "Python (cofkit)"],
        cwd=repo_root,
        check=True,
    )
    print("cofkit 环境和 Jupyter 内核已准备完成。")

In [ ]:
# Conda 没有用于创建环境的稳定公共 Python API。
# 因此，上面的可运行单元格通过 subprocess 参数安全地调用 Conda 和 pip。

如果本 Notebook 是用其他内核打开的，现在可以从 Jupyter 的内核菜单中选择 **Python (cofkit)**。后续可执行单元格都会显式使用 `conda run -n cofkit`，因此在这里切换内核并非必需。Notebooks 2–4 已将 `cofkit` 声明为默认内核。

## 验证安装

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

repo_root = Path.cwd().resolve()
conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
if not conda:
    raise RuntimeError("未找到 Conda。请先运行环境设置单元格。")

commands = [
    [conda, "run", "-n", "cofkit", "cofkit", "--version"],
    [conda, "run", "-n", "cofkit", "python", "-c",
     "import cofkit; print('Python import:', cofkit.__version__)"],
    [conda, "run", "-n", "cofkit", "cofkit", "build", "list-templates"],
]
for command in commands:
    subprocess.run(command, cwd=repo_root, check=True)

In [ ]:
# Python 等价实现（已注释）：验证软件包能够导入。
# import cofkit
# print(cofkit.__version__)
#
# from cofkit import ReactionLibrary
# library = ReactionLibrary.builtin()
# print(library)

## 下一步

环境已准备完成。继续学习 [Notebook 2](02_first_cof_build.ipynb)，构建第一个 COF。